# データ理解

In [1]:
!pip install optuna optuna-integration

# import lightgbm as lgb
import optuna.integration.lightgbm as lgb
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd

from sklearn import metrics
import matplotlib.pyplot as plt

import optuna

/Users/shigeyuki-t/anaconda3/lib/python3.7/site-packages/dask/config.py:168: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  data = yaml.load(f.read()) or {}
/Users/shigeyuki-t/anaconda3/lib/python3.7/site-packages/distributed/config.py:20: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  defaults = yaml.load(f)


In [2]:
file_path = '/Users/shigeyuki-t/Desktop/FirstExperiment/分析用/05.csv'
dataset = pd.read_csv(file_path)
dataset.head(3)

,Unnamed: 0,sensor_id,data_time,QuestionNum,choice_left,choice_right,event,isAnswerCorrect,response_time,response_time_ave,...,scroll_count,scroll_length,scroll_speed,scroll_time,tap_count,tap_interval,tap_position_x,tap_position_y,taskNum,view_position
0,0,suke.kazuma@gmail.com,2024-05-07 03:33:51,0,0,0,scroll,True,0.0,0.0,...,2,0.0,0.0,0.5,0,0.0,0.0,0.0,1,513.0
1,1,suke.kazuma@gmail.com,2024-05-07 03:33:50,0,0,0,scroll,True,0.0,0.0,...,1,935.0,9350.0,0.1,0,0.0,0.0,0.0,1,513.0
2,2,suke.kazuma@gmail.com,2024-05-07 03:33:55,0,1,0,response,True,0.0,0.0,...,2,0.0,0.0,0.5,1,0.0,0.0,0.0,1,513.0


In [3]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17872 entries, 0 to 17871
Data columns (total 22 columns):
Unnamed: 0           17872 non-null int64
sensor_id            17872 non-null object
data_time            17872 non-null object
QuestionNum          17872 non-null int64
choice_left          17872 non-null int64
choice_right         17872 non-null int64
event                17872 non-null object
isAnswerCorrect      17872 non-null bool
response_time        17872 non-null float64
response_time_ave    17872 non-null float64
screen_height        17872 non-null int64
screen_width         17872 non-null int64
scroll_count         17872 non-null int64
scroll_length        17872 non-null float64
scroll_speed         17872 non-null float64
scroll_time          17872 non-null float64
tap_count            17872 non-null int64
tap_interval         17872 non-null float64
tap_position_x       17872 non-null float64
tap_position_y       17872 non-null float64
taskNum              17872 non-nu

In [4]:
dataset.describe()

,Unnamed: 0,QuestionNum,choice_left,choice_right,response_time,response_time_ave,screen_height,screen_width,scroll_count,scroll_length,scroll_speed,scroll_time,tap_count,tap_interval,tap_position_x,tap_position_y,taskNum,view_position
count,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000,17872.000000
mean,8935.500000,77.161929,33.562892,31.261470,6.052865,11.362357,835.017122,392.941137,145.515947,305.285813,221.803827,4.667329,66.378973,5.644231,135.621505,191.429124,65.502238,29910.213153
std,5159.346341,44.273359,23.761142,21.657783,20.016043,8.647612,101.921501,18.623963,105.075389,1461.489288,2732.775480,10.639696,44.879529,18.626745,153.908233,213.912419,44.065405,19852.965286
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,600.000000,320.000000,1.000000,0.000000,0.000000,0.100000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,4467.750000,40.000000,13.000000,12.000000,0.400000,7.367380,812.000000,375.000000,60.000000,23.666667,16.289386,0.900000,27.000000,0.400000,0.000000,0.000000,27.000000,12606.916667
50%,8935.500000,78.000000,30.000000,28.000000,2.200000,10.573256,844.000000,390.000000,127.000000,241.500000,58.333333,2.300000,61.000000,2.100000,36.333328,27.000000,60.000000,28231.250000
75%,13403.250000,117.000000,52.000000,49.000000,6.400000,12.513131,852.000000,414.000000,211.000000,388.000000,141.190476,5.500000,103.000000,6.200000,308.000000,417.500000,102.000000,46588.000000
max,17871.000000,151.000000,114.000000,89.000000,886.400000,114.666667,1194.000000,430.000000,484.000000,63917.666667,242237.500000,873.500000,159.000000,886.700000,531.500000,657.333328,151.000000,89287.500000


# データ成形

In [5]:
# csvファイルの作成
specific_mail_dataset = dataset[dataset['sensor_id'] == 'ishiyama.rin.ip5@naist.ac.jp']
csv_create_location = '/Users/shigeyuki-t/Desktop/FirstExperiment/分析用/ishiyama.rin.ip5@naist.ac.jp.csv'
specific_mail_dataset.to_csv(csv_create_location, index = True)
specific_mail_dataset.describe()

,Unnamed: 0,QuestionNum,choice_left,choice_right,response_time,response_time_ave,screen_height,screen_width,scroll_count,scroll_length,scroll_speed,scroll_time,tap_count,tap_interval,tap_position_x,tap_position_y,taskNum,view_position
count,399.000000,399.000000,399.000000,399.000000,399.000000,399.000000,399.0,399.0,399.000000,399.000000,399.000000,399.000000,399.000000,399.000000,399.000000,399.000000,399.000000,399.000000
mean,761.799499,81.994987,42.914787,31.418546,5.473183,11.563851,844.0,390.0,118.045113,464.298246,292.409768,7.262406,76.045113,5.316291,282.012529,300.959895,74.957393,31985.738513
std,887.026173,46.268552,26.313013,18.605472,7.637606,2.779069,0.0,0.0,69.377838,1983.178411,2331.620488,8.197045,45.855661,7.561946,123.535062,165.379348,44.763821,19317.845103
min,129.000000,0.000000,0.000000,0.000000,0.000000,0.000000,844.0,390.0,1.000000,0.000000,0.000000,0.100000,0.000000,0.000000,0.000000,0.000000,1.000000,35.000000
25%,428.500000,42.000000,17.000000,16.000000,0.400000,10.573256,844.0,390.0,60.500000,126.666667,24.024483,1.550000,34.000000,0.400000,315.000000,250.666656,33.500000,14652.333333
50%,645.000000,84.000000,44.000000,30.000000,2.500000,10.953247,844.0,390.0,119.000000,366.333333,49.593496,4.100000,75.000000,2.400000,315.000000,250.666656,75.000000,31249.333333
75%,943.000000,127.000000,67.500000,48.000000,6.700000,13.128205,844.0,390.0,175.000000,399.333333,112.592593,10.000000,118.500000,6.000000,353.333328,472.000000,116.000000,50242.000000
max,7616.000000,151.000000,84.000000,66.000000,47.400000,20.000000,844.0,390.0,240.000000,34295.333333,38934.000000,41.400000,153.000000,47.500000,366.333328,472.000000,151.000000,64350.333333


In [9]:
# データの読み込み
data_clean = pd.read_csv('/Users/shigeyuki-t/Desktop/FirstExperiment/分析用/datacheck.csv')

# True = 1, False = 0, レスポンス以外のときの値が-1
data_clean.loc[data_clean['event'] != 'response', 'isAnswerCorrect'] = np.nan
data_clean['isAnswerCorrect'].fillna(-1, inplace=True)

# data_timeをdatetimeに変換してソート
data_clean['data_time'] = pd.to_datetime(data_clean['data_time'])
data_clean = data_clean.sort_values(by='data_time')

# data_time の差分を秒単位で計算し、response_time に代入
data_clean.loc[data_clean['taskNum'] == 1, 'response_time'] = data_clean['data_time'].diff().dt.total_seconds().shift(-1)

# カテゴリ変数をダミー変数に変換
data_clean = pd.get_dummies(data_clean, columns=['event'])

#　各スクロールの組を示す変数
data_clean['scroll_group'] = 0

# 初期グループ番号
group_number = 1

# 前の行の値を保存する変数
previous_value = data_clean.iloc[0]['scroll_time']
data_clean.at[0, 'scroll_group'] = group_number

# グループ分けのループ
for index in range(1, len(data_clean)):
    current_value = data_clean.iloc[index]['scroll_time']
    if current_value >= previous_value:
        data_clean.at[index, 'scroll_group'] = group_number
    else:
        group_number += 1
        data_clean.at[index, 'scroll_group'] = group_number
    previous_value = current_value

# scroll_timeが複数のtaskNumに跨って継続しているときの処理
max_task_num = data_clean['taskNum'].max()
for task_num in range(max_task_num):
    current_task_group = data_clean[data_clean['taskNum']==task_num]
    next_task_group = data_clean[data_clean['taskNum']==task_num+1]
    # np.intersect1dは複数配列の共通項を取り出す
    common_groups = np.intersect1d(current_task_group['scroll_group'].unique(),next_task_group['scroll_group'].unique())
    
    for group in common_groups:
        last_scroll_time_current = current_task_group[current_task_group['scroll_group'] == group]['scroll_time'].values[-1]
        data_clean.loc[(data_clean['taskNum'] == task_num + 1)&(data_clean['scroll_group'] == group),'scroll_time'] -= last_scroll_time_current
    

# # taskNum 内で scroll_group と scroll_time に基づいて降順ソート
# data_clean = data_clean.sort_values(by=['taskNum', 'scroll_group', 'scroll_time'], ascending=[True, True, False])

# # グループごとにscroll_timeを降順にソートし、元のインデックスを保持
# sorted_df = data_clean.sort_values(by=['scroll_group', 'scroll_time'], ascending=[True, False])
# sorted_df = sorted_df.reset_index(drop=True)

# # 次の行の値が現在の行の値より小さい場合はそのまま、それ以外は0に置換　＝スクロール中の値は0にする
# data_clean['scroll_time'] = data_clean.apply(lambda row: row['scroll_time'] if (row.name < len(data_clean) - 1 and data_clean.at[row.name + 1, 'scroll_time'] < row['scroll_time']) else 0, axis=1)

# # 結果を元のデータフレームに適用
# data_clean['sorted_scroll_time'] = sorted_df['scroll_time']

data_clean.head(50)

,Unnamed: 0,Unnamed: 0.1,sensor_id,data_time,QuestionNum,choice_left,choice_right,isAnswerCorrect,response_time,response_time_ave,...,scroll_time,tap_count,tap_interval,tap_position_x,tap_position_y,taskNum,view_position,event_response,event_scroll,scroll_group
1,1,1,suke.kazuma@gmail.com,2024-05-07 03:33:50,0,0,0,-1.0,1.0,0.000000,...,0.1,0,0.0,0.0,0.0,1,513.000000,0,1,1
0,0,0,suke.kazuma@gmail.com,2024-05-07 03:33:51,0,0,0,-1.0,4.0,0.000000,...,0.5,0,0.0,0.0,0.0,1,513.000000,0,1,1
2,2,2,suke.kazuma@gmail.com,2024-05-07 03:33:55,0,1,0,1.0,0.0,0.000000,...,0.5,1,0.0,0.0,0.0,1,513.000000,1,0,1
3,3,3,suke.kazuma@gmail.com,2024-05-07 03:33:55,120,1,0,-1.0,0.0,0.000000,...,4.0,1,0.1,0.0,0.0,2,513.000000,0,1,1
4,4,4,suke.kazuma@gmail.com,2024-05-07 03:33:56,120,1,0,-1.0,0.4,0.000000,...,0.3,1,0.4,0.0,0.0,2,513.000000,0,1,2
5,5,5,suke.kazuma@gmail.com,2024-05-07 03:33:57,120,1,0,-1.0,1.6,0.000000,...,1.2,1,1.7,0.0,0.0,2,353.333333,0,1,2
6,6,6,suke.kazuma@gmail.com,2024-05-07 03:33:58,120,1,0,-1.0,2.4,0.000000,...,0.7,1,2.5,0.0,0.0,2,371.666667,0,1,3
7,7,7,suke.kazuma@gmail.com,2024-05-07 03:33:59,120,1,0,-1.0,4.1,0.000000,...,1.6,1,4.1,0.0,0.0,2,371.666667,0,1,3
8,8,8,suke.kazuma@gmail.com,2024-05-07 03:34:03,120,2,0,1.0,7.7,3.850000,...,1.6,2,7.7,0.0,0.0,2,371.666667,1,0,3
9,9,9,suke.kazuma@gmail.com,2024-05-07 03:34:03,51,2,0,-1.0,0.4,3.850000,...,2.4,2,0.4,0.0,0.0,3,245.000000,0,1,3


In [7]:
# taskNumが同じ行を１行にまとめる
# 集計方法を指定
aggregation_functions = {
    "choice_left": "max", 
    "choice_right": "max",
    "isAnswerCorrect": "max",
    "response_time": "max",
#     "response_time_ave": "mean",
    "screen_height": "max",
    "screen_width": "max",
    "scroll_count": "max",
    
    "scroll_length": "sum",
#     "scroll_speed": "mean",
    "scroll_time": "sum",
#     "sorted_scroll_time":"sum",
    
    "tap_count": "max",
#     "tap_interval": "mean",
    "tap_position_x": "max",
    "tap_position_y": "max",
    "view_position": "last",
    "event_response": "sum",
    "event_scroll": "sum" 
}

# グループ化して集計
df = data_clean .groupby("taskNum").agg(aggregation_functions).reset_index()

# 蓄積値を増加値に変更
df['s_choice_left'] = df['choice_left'].diff().fillna(df['choice_left'])
df['s_choice_right'] = df['choice_right'].diff().fillna(df['choice_right'])
df['s_tap_count'] = df['tap_count'].diff().fillna(df['tap_count'])
df['s_view_position'] = df['view_position'].diff().fillna(df['view_position'])

df = df.drop(columns=['choice_left','choice_right','tap_count','scroll_count','event_response','view_position','s_choice_right'	])

df['s_scroll_speed'] = df['scroll_length']/df['scroll_time']
csv_create_location = '/Users/shigeyuki-t/Desktop/FirstExperiment/分析用/data０６２４.csv'
df.to_csv(csv_create_location, index = True)

df.head(10)


,taskNum,isAnswerCorrect,response_time,screen_height,screen_width,scroll_length,scroll_time,tap_position_x,tap_position_y,event_scroll,s_choice_left,s_tap_count,s_view_position,s_scroll_speed
0,1,1.0,4.0,844,390,935.000000,1.1,0.0,0.0,2,1.0,1.0,513.000000,850.000000
1,2,1.0,7.7,844,390,162.333333,9.4,0.0,0.0,5,1.0,1.0,-141.333333,17.269504
2,3,1.0,3.6,844,390,616.666667,4.6,0.0,0.0,2,1.0,1.0,-344.000000,134.057971
3,4,1.0,2.1,844,390,697.333333,4.6,0.0,0.0,1,1.0,1.0,348.666667,151.594203
4,5,1.0,4.4,844,390,356.333333,2.8,0.0,0.0,2,1.0,1.0,349.666667,127.261905
5,6,1.0,3.6,844,390,1062.000000,5.7,0.0,0.0,2,1.0,1.0,608.000000,186.315789
6,7,0.0,5.4,844,390,772.666667,3.2,0.0,0.0,1,0.0,1.0,386.333333,241.458333
7,8,1.0,4.4,844,390,526.000000,4.6,0.0,0.0,2,1.0,1.0,350.333333,114.347826
8,9,1.0,5.6,844,390,766.000000,7.0,0.0,0.0,1,1.0,1.0,383.000000,109.428571
9,10,0.0,5.9,844,390,409.666667,2.7,0.0,0.0,2,0.0,1.0,382.333333,151.728395


# モデル設計

In [8]:
# # taskNumが1のデータ
# train_data = data_clean[data_clean['taskNum'] == 1]
# X_train = data_clean.drop(columns=['isAnswerCorrect'])
# y_train = data_clean['isAnswerCorrect']

# # taskNumが2のデータ
# test_data = data_clean[data_clean['taskNum'] == 2]
# X_test = test_data.drop(columns=['isAnswerCorrect'])
# y_test = test_data['isAnswerCorrect']



# # 全てのカラムをfloatに変換
# data_clean = data_clean.astype(float)

# # LightGBM用データセットの作成
# train_dataset = lgb.Dataset(X_train, label=y_train)
# test_dataset = lgb.Dataset(X_test, label=y_test, reference=train_dataset)

# # モデルの設定
# params = {
#     'objective': 'binary',
#     'metric':'auc',
# }

# # モデルの作成と訓練
# model = lgb.train(params,train_dataset,num_boost_round=1000,valid_sets=test_dataset )

# # 保存
# model.save_model('model.txt')

# # 予測
# y_pred = model.predict(X_test, num_iteration=model.best_iteration)

# # AUC (Area Under the Curve) を計算する
# fpr, tpr, thresholds = metrics.roc_curve(y_test,y_pred)
# auc = metrics.auc(fpr, tpr)
# print(auc)

# # ROC曲線をプロット
# plt.plot(fpr, tpr, label='ROC curve (area = %.2f)'%auc)
# plt.legend()
# plt.title('ROC curve')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.grid(True)

# # 特徴量の重要度出力
# print(model.feature_importance())

# # 特徴量の重要度をプロット
# lgb.plot_importance(model)

TypeError: cannot astype a datetimelike from [datetime64[ns]] to [float64]

In [ ]:
# print(dataset['isAnswerCorrect'].value_counts())
# Y = dataset['isAnswerCorrect']
# Y = dataset['isAnswerCorrect'].replace({True: 1, False: 0})
# print(Y.value_counts())

# X = dataset.drop(columns=['isAnswerCorrect','sensor_id','data_time','event','Unnamed: 0'],axis = 1)
# print(X.columns)

# X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.3,random_state=0,shuffle=True)
# lgb_train=lgb.Dataset(X_train,Y_train)
# lgb_eval=lgb.Dataset(X_test,Y_test,reference=lgb_train)

In [ ]:
# from lightgbm import LGBMRegressor
# params = {
#     'objective': 'binary',
#     'metrics':'auc',
# #     'boosting_type':'gbdt',
# #     'verbosity': -1,
# #     'random_state': 42,
# #     'learning_rate': 0.01,
# #     'verbose_eval':50,
# #     'early_stopping_rounds':100
# }
# print("タイプ：",type(lgb_train))
# model = lgb.train(params,lgb_train,num_boost_round=1000,valid_sets=lgb_eval)
# model.save_model('model.txt')
# print(model.params)

In [ ]:
# # AUC (Area Under the Curve) を計算する
# Y_pred = model.predict(X_test, num_iteration=model.best_iteration)
# fpr, tpr, thresholds = metrics.roc_curve(Y_test,Y_pred)
# auc = metrics.auc(fpr, tpr)
# print(auc)

# # ROC曲線をプロット
# plt.plot(fpr, tpr, label='ROC curve (area = %.2f)'%auc)
# plt.legend()
# plt.title('ROC curve')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.grid(True)

# # 特徴量の重要度出力
# print(model.feature_importance())

# # 特徴量の重要度をプロット
# lgb.plot_importance(model)